# Chunking and Embeddings: A Self-Contained Pipeline Demo (Optional)

> **Run this in Amazon SageMaker AI.** This notebook is designed to run in Amazon SageMaker AI Studio. Running it locally (for example in VS Code) or in another environment is not supported and will fail. Complete the [Environment Setup: Amazon SageMaker AI](https://neo4j-partners.github.io/neo4j-sec-filings-graphrag-workshop/workshop/neo4j-sec-filings-graphrag-workshop/1.0/part2-setup-instructions.html) steps first to launch SageMaker AI Studio, then open this notebook there.

**This lab is optional but recommended.** Lab 1 already loaded the complete knowledge graph: structured entities, document chunks, embeddings, and the vector index. Everything downstream is ready to run without it, so nothing here is a prerequisite for Labs 3 onward. Run it anyway to see *how* that chunk, embed, index, and search pipeline actually works.

To keep the real graph safe, everything here is built in an isolated sandbox. Every demo node carries a `:Demo` label plus its own `Demo*` label, and the vector index is named `demoChunkEmbeddings`. Nothing touches the `:Company`, `:Chunk`, or `chunkEmbeddings` data that Lab 1 loaded. The final cell deletes the sandbox, so you can run this notebook as many times as you like.

**Learning Objectives:**
- Split filing text into overlapping chunks
- Generate embeddings with Amazon Titan Text Embeddings V2
- Store chunks, embeddings, and their graph relationships in Neo4j
- Create a vector index and run a semantic search

## The Pipeline You Will Build

A GraphRAG pipeline connects **unstructured** text (document chunks) to **structured** entities:

```
(:DemoCompany)-[:OFFERS]->(:DemoProduct)
(:DemoCompany)-[:FILED]->(:DemoDocument)<-[:FROM_DOCUMENT]-(:DemoChunk)-[:NEXT_CHUNK]->(:DemoChunk)
(:DemoProduct)-[:FROM_CHUNK]->(:DemoChunk)
```

A vector search finds relevant chunks, then graph traversal reaches the connected entities. All of it uses `Demo*` labels so it stays isolated from the real graph.

In [ ]:
%pip install "neo4j-graphrag[bedrock]>=1.18.0" -q

In [ ]:
import json
import os

from lib.data_utils import split_text, get_embedder
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()
print('Connected to Neo4j!')
print(f'Embedder: {embedder.model_id}')

## Reset the Sandbox

Remove any demo nodes and the demo index left over from a previous run. This targets only `:Demo` nodes and the `demoChunkEmbeddings` index, so the graph Lab 1 loaded is never touched. The same function runs again at the end to tear everything down.

In [ ]:
DEMO_INDEX = 'demoChunkEmbeddings'


def reset_sandbox(driver):
    """Delete only the demo sandbox: :Demo nodes and the demo vector index."""
    with driver.session() as session:
        deleted = session.run(
            'MATCH (n:Demo) DETACH DELETE n RETURN count(n) AS deleted'
        ).single()['deleted']
        session.run(f'DROP INDEX {DEMO_INDEX} IF EXISTS')
    print(f'Sandbox reset: removed {deleted} demo node(s) and dropped {DEMO_INDEX} if present.')


def count_real_nodes(driver):
    """Count nodes that are not part of the demo sandbox."""
    with driver.session() as session:
        return session.run('MATCH (n) WHERE NOT n:Demo RETURN count(n) AS n').single()['n']


reset_sandbox(driver)

# Record the real (non-demo) node count so the teardown can prove the sandbox
# left it unchanged.
baseline_nodes = count_real_nodes(driver)
print(f'Real graph has {baseline_nodes} node(s); the sandbox will not change this count.')

## Load the Sample Filing

`financial_data.json` holds a small slice of Apple's 10-K: company metadata, a handful of products, and the filing text to chunk and embed.

In [ ]:
with open('financial_data.json', 'r') as f:
    filing_data = json.load(f)

company = filing_data['company']
products = filing_data['products']
document = filing_data['document']
filing_text = filing_data['filing_text']

print(f'Company: {company["name"]} ({company["ticker"]})')
print(f'Products: {len(products)}')
print(f'Filing text: {len(filing_text)} characters')
print(f'\nPreview:\n{filing_text[:300]}...')

## Step 1: Split the Text into Chunks

Embedding models work best on short passages, so the filing is split into 500-character chunks with 50 characters of overlap. The overlap keeps a sentence that straddles a boundary intact in at least one chunk.

In [ ]:
chunks = split_text(filing_text)
print(f'Split into {len(chunks)} chunks\n')
for i, chunk in enumerate(chunks):
    print(f'Chunk {i} ({len(chunk)} chars): {chunk[:80]}...')

## Step 2: Generate Embeddings

An embedding turns text into a vector that captures meaning. Titan Text Embeddings V2 returns a 1024-dimensional vector, and chunks about similar topics point in similar directions. Start with a single embedding to see the shape of the output.

In [ ]:
sample = "Apple's iPhone revenue grew year over year"
vector = embedder.embed_query(sample)

print(f'Text: "{sample}"')
print(f'Dimensions: {len(vector)}')
print(f'First 5 values: {vector[:5]}')

Now embed every chunk. Each chunk's vector is kept in order so it can be stored on the matching node in the next step.

In [ ]:
chunk_embeddings = [embedder.embed_query(text) for text in chunks]

assert len(chunk_embeddings) == len(chunks), 'expected one embedding per chunk'
assert all(len(v) == 1024 for v in chunk_embeddings), 'Titan V2 embeddings must be 1024-dimensional'

print(f'Generated {len(chunk_embeddings)} embeddings of {len(chunk_embeddings[0])} dimensions each.')

## Step 3: Build the Graph Pipeline

Store the pipeline in Neo4j under the sandbox labels:
- a `:DemoDocument` for the filing, linked to a `:DemoCompany` with `FILED`
- one `:DemoChunk` per chunk (text and embedding), linked to the document with `FROM_DOCUMENT` and chained in reading order with `NEXT_CHUNK`
- `:DemoProduct` nodes linked to the company with `OFFERS` and to the chunks that mention them with `FROM_CHUNK`

Chunk text and structure are written first, then `upsert_vectors` stores each embedding on its node.

In [ ]:
from neo4j_graphrag.indexes import upsert_vectors


def build_pipeline(driver, company, document, products, chunks):
    """Build the sandbox graph and return chunk element ids in index order."""
    doc_name = document['name']
    chunk_rows = [{'index': i, 'text': t} for i, t in enumerate(chunks)]

    with driver.session() as session:
        # Structured layer: company, document, products, and their cross-links
        session.run(
            """
            MERGE (c:DemoCompany:Demo {name: $name})
              SET c.ticker = $ticker
            MERGE (d:DemoDocument:Demo {name: $doc_name})
            MERGE (c)-[:FILED]->(d)
            WITH c
            UNWIND $products AS prod
            MERGE (p:DemoProduct:Demo {name: prod.name})
              SET p.description = prod.description
            MERGE (c)-[:OFFERS]->(p)
            """,
            name=company['name'], ticker=company['ticker'],
            doc_name=doc_name, products=products,
        )

        # Unstructured layer: chunk nodes linked to the document
        session.run(
            """
            MATCH (d:DemoDocument:Demo {name: $doc_name})
            UNWIND $chunks AS chunk
            MERGE (ch:DemoChunk:Demo {index: chunk.index})
              SET ch.text = chunk.text
            MERGE (ch)-[:FROM_DOCUMENT]->(d)
            """,
            doc_name=doc_name, chunks=chunk_rows,
        )

        # Preserve reading order with NEXT_CHUNK
        session.run(
            """
            UNWIND range(0, $n - 2) AS i
            MATCH (a:DemoChunk:Demo {index: i})
            MATCH (b:DemoChunk:Demo {index: i + 1})
            MERGE (a)-[:NEXT_CHUNK]->(b)
            """,
            n=len(chunks),
        )

        # Cross-link products to the chunks that mention them
        session.run(
            """
            MATCH (p:DemoProduct:Demo)
            MATCH (ch:DemoChunk:Demo)
            WHERE ch.text CONTAINS p.name
            MERGE (p)-[:FROM_CHUNK]->(ch)
            """
        )

        # Return chunk element ids in index order for the embedding upsert
        result = session.run(
            """
            MATCH (ch:DemoChunk:Demo)
            RETURN elementId(ch) AS id
            ORDER BY ch.index
            """
        )
        return [record['id'] for record in result]


chunk_ids = build_pipeline(driver, company, document, products, chunks)

upsert_vectors(
    driver,
    ids=chunk_ids,
    embedding_property='embedding',
    embeddings=chunk_embeddings,
)

assert len(chunk_ids) == len(chunks), 'every chunk should have a node'
with driver.session() as session:
    embedded = session.run(
        'MATCH (c:DemoChunk:Demo) WHERE c.embedding IS NOT NULL RETURN count(c) AS n'
    ).single()['n']
assert embedded == len(chunks), f'expected {len(chunks)} embedded chunks, found {embedded}'

print(f'Built sandbox graph and stored {embedded} chunk embeddings.')

## Step 4: Create the Vector Index

The index enables fast approximate nearest-neighbor search over the chunk embeddings. It uses a sandbox-specific name so it never collides with Lab 1's `chunkEmbeddings` index.

In [ ]:
from neo4j_graphrag.indexes import create_vector_index

create_vector_index(
    driver,
    name=DEMO_INDEX,
    label='DemoChunk',
    embedding_property='embedding',
    dimensions=1024,
    similarity_fn='cosine',
)

print(f'Created vector index {DEMO_INDEX}')

with driver.session() as session:
    indexes = {
        r['name']: r
        for r in session.run('SHOW VECTOR INDEXES YIELD name, labelsOrTypes, properties, state')
    }

assert DEMO_INDEX in indexes, f'{DEMO_INDEX} was not created'
record = indexes[DEMO_INDEX]
print(f'  {record["name"]} on {record["labelsOrTypes"]}.{record["properties"]} ({record["state"]})')

## Step 5: Run a Semantic Search

`VectorRetriever` embeds a question and returns the most similar chunks. This confirms the pipeline works end to end. Lab 3 builds on this with graph-enriched retrieval that also returns the connected entities.

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever

retriever = VectorRetriever(
    driver=driver,
    index_name=DEMO_INDEX,
    embedder=embedder,
    return_properties=['text'],
)

query = "What products does Apple sell?"
print(f'Query: "{query}"\n')

results = retriever.search(query_text=query, top_k=3)

assert results.items, 'vector search returned no results'

for i, item in enumerate(results.items, 1):
    score = item.metadata.get('score', 0)
    content = item.content if isinstance(item.content, str) else str(item.content)
    print(f'{i}. score={score:.4f}')
    print(f'   {content[:150]}...\n')

## Step 6: Tear Down the Sandbox

Remove everything this notebook created so the database returns to exactly the graph Lab 1 loaded. Because every demo node carries the `:Demo` label and the index is `demoChunkEmbeddings`, cleanup is a single targeted delete plus an index drop.

In [ ]:
reset_sandbox(driver)

with driver.session() as session:
    remaining = session.run('MATCH (n:Demo) RETURN count(n) AS n').single()['n']

assert remaining == 0, f'{remaining} demo node(s) still present after teardown'
assert count_real_nodes(driver) == baseline_nodes, 'the real graph node count changed'

print('Sandbox removed. The graph from Lab 1 is untouched.')

## Summary

You built a complete GraphRAG data pipeline in an isolated sandbox:

1. **Chunking** split the filing into overlapping passages
2. **Embeddings** turned each chunk into a 1024-dimensional Titan vector
3. **Graph pipeline** stored chunks, embeddings, and their relationships alongside the structured company, product, and document entities
4. **Vector index** enabled semantic search, verified with `VectorRetriever`

Everything ran under `:Demo` labels and was torn down at the end, so the graph Lab 1 loaded is unchanged. Continue to Lab 3 to run retrieval over that full graph.

---

**Next:** [Lab 3 - Semantic Search and GraphRAG](../Lab_3_GraphRAG_Search/01_vector_retriever.ipynb)

In [ ]:
driver.close()
print('Connection closed.')